# Exercise 1.1 — Word Embeddings: words as numbers
**Created by Shivank Agarwal**

A computer cannot read the word *king*. It can only work with numbers.
An **embedding** is a list of numbers that stands for a word. Words with a similar meaning get similar lists.

In this notebook you will:
1. Load a ready-made embedding model (GloVe, 50 numbers per word)
2. Look at the vector for a word
3. Do "word maths" like **king − man + woman ≈ queen**
4. Measure how close two words are with **cosine similarity** (the same maths every RAG system uses)

> The first run downloads the model (about 66 MB). After that it loads from your computer.

In [1]:
# Libraries:
#   numpy  -> maths on lists of numbers (vectors)
#   gensim -> gives us ready-made word vectors (GloVe)
import numpy as np
import gensim.downloader as api

# "glove-wiki-gigaword-50" = GloVe vectors trained on Wikipedia + news text.
# Every word in its vocabulary (400,000 words) becomes a list of 50 numbers.
model = api.load("glove-wiki-gigaword-50")
print("Words in vocabulary:", len(model.key_to_index))

Words in vocabulary: 400000


## Step 1 — What does a word look like to the computer?

In [3]:
# ----- TODO 1 (solved): Get the vector for the word "king" and print its shape and first 5 numbers -----
v_king = model["king"]
print("Shape of the 'king' vector:", v_king.shape)  # expect (50,)
print("First 5 numbers:", v_king[:5])

Shape of the 'king' vector: (50,)
First 5 numbers: [ 0.50451   0.68607  -0.59517  -0.022801  0.60046 ]


## Step 2 — Which words live close to "king"?
`most_similar` finds the words whose vectors point in the most similar direction.

In [4]:
for word, score in model.most_similar("king", topn=5):
    print(f"{word:<10} {score:.3f}")

prince     0.824
queen      0.784
ii         0.775
emperor    0.774
son        0.767


## Step 3 — Word maths (analogies)
If *king* is to *man* as ? is to *woman*, then **king − man + woman** should land near **queen**.

⚠️ Gotcha: if you build the vector yourself and search with it, the closest word is often one of the
words you started with (for example *king* itself). `most_similar(positive=[...], negative=[...])`
does the same maths **and** skips the input words. That is why we use it.

In [5]:
# ----- TODO 2 (solved): Use word maths to answer "king - man + woman = ?" -----
answer = model.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
print("king - man + woman =", answer)

king - man + woman = [('queen', 0.8523604273796082)]


In [6]:
# The same maths done "by hand" with numpy, to see the gotcha for yourself
new_vector = model["king"] - model["man"] + model["woman"]
print("Raw vector search (input words NOT removed):", model.similar_by_vector(new_vector, topn=3))

Raw vector search (input words NOT removed): [('king', 0.8859834671020508), ('queen', 0.8609582185745239), ('daughter', 0.7684512138366699)]


In [18]:
model.similar_by_vector(model["india"] - model["bengali"], topn=5)

[('hopes', 0.6913630962371826),
 ('bid', 0.6531022191047668),
 ('korea', 0.6466168761253357),
 ('united', 0.6212124824523926),
 ('ahead', 0.6201883554458618)]

In [16]:
# ----- TODO 3 (solved): Capital cities: "paris - france + india = ?" -----
answer_2 = model.most_similar(positive=["paris", "india"], negative=["france"], topn=1)
print("paris - france + india =", answer_2)

paris - france + india = [('delhi', 0.8888585567474365)]


## Step 4 — Cosine similarity by hand
Cosine similarity tells us how much two vectors point the same way:

`cosine(a, b) = (a · b) / (|a| × |b|)`

* close to **1** → very similar meaning
* close to **0** → unrelated

You will use this exact formula again in the next exercise to search documents.

In [19]:
# ----- TODO 4 (solved): Write a cosine_similarity(a, b) function using numpy -----
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = [("king", "queen"), ("cricket", "football"), ("king", "banana")]
for w1, w2 in pairs:
    mine = cosine_similarity(model[w1], model[w2])
    library = model.similarity(w1, w2)  # gensim's own answer, to check yours
    print(f"{w1:>8} vs {w2:<9} -> my answer: {mine:.3f} | gensim: {library:.3f}")

    king vs queen     -> my answer: 0.784 | gensim: 0.784
 cricket vs football  -> my answer: 0.696 | gensim: 0.696
    king vs banana    -> my answer: 0.221 | gensim: 0.221
